In [1]:
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, ".")  # adjust if ethan_original/ isn't next to your notebook
from data_preprocessing import DataProcessor
from dnn_model import DNNModel

KAGGLE_PATH = "../../data/kaggle_dataset.csv"
ETHAN_CLEAN_PATH = "../../data/bitcoin_for_weka.csv"
BIGQUERY_PATH = "../../data/real_bitcoin_blocks_raw.csv"

In [2]:
def run_experiment(csv_path, label):
    print(f"\n{'='*60}\n{label}\n{'='*60}")

    dp = DataProcessor(csv_path)
    df = dp.load_data()
    miners_df = dp.create_miner_simulation(df)
    X_train, X_test, y_train, y_test, features = dp.prepare_training_data(miners_df)

    # Path A: his own train() + evaluate() — the partial_fit loop
    model_a = DNNModel(input_shape=X_train.shape[1])
    model_a.train(X_train, y_train, epochs=100, batch_size=16)
    test_acc = model_a.evaluate(X_test, y_test)

    # Path B: his own cross_validate() on the full data (train+test recombined)
    X_full = np.vstack([X_train, X_test])
    y_full = np.concatenate([y_train, y_test])
    model_b = DNNModel(input_shape=X_train.shape[1])
    cv_scores = model_b.cross_validate(X_full, y_full, cv=5)

    print(f"\n>>> {label}: Path A test acc = {test_acc:.4f} | Path B mean CV = {cv_scores.mean():.4f}")
    return test_acc, cv_scores.mean()

In [3]:
def prepare_bigquery_csv(src_path, dst_path):
    df = pd.read_csv(src_path)
    df = df.rename(columns={
        "block_number": "height",
        "transaction_count": "tx_count",
        "total_output_satoshis_excl_coinbase": "output_amount",
    })
    df.to_csv(dst_path, index=False)

prepare_bigquery_csv(BIGQUERY_PATH, "bigquery_renamed_temp.csv")

In [4]:
results = {}
results["Kaggle"] = run_experiment(KAGGLE_PATH, "Kaggle-format dataset")
results["Ethan cleaned"] = run_experiment(ETHAN_CLEAN_PATH, "Ethan's cleaned dataset")
results["BigQuery"] = run_experiment("bigquery_renamed_temp.csv", "BigQuery dataset")


Kaggle-format dataset
Loaded dataset with shape: (810909, 13)
Columns: ['height', 'timestamp', 'size', 'tx_count', 'difficulty', 'median_fee_rate', 'avg_fee_rate', 'total_fees', 'fee_range_min', 'fee_range_max', 'input_count', 'output_count', 'output_amount']
Features used: ['blocks_mined', 'avg_transactions', 'avg_volume', 'difficulty', 'avg_fee', 'fee_volatility', 'avg_block_size', 'age', 'profitability']
Feature statistics:
       blocks_mined  avg_transactions    avg_volume    difficulty  \
count    100.000000        100.000000    100.000000  1.000000e+02   
mean    8109.090000       1114.228251  10404.839399  7.821774e+12   
std        0.287623          6.449839    354.259323  1.976015e+09   
min     8109.000000       1099.984708   9702.485099  7.817118e+12   
25%     8109.000000       1110.411148  10135.084353  7.820612e+12   
50%     8109.000000       1114.179060  10369.657438  7.821835e+12   
75%     8109.000000       1118.075256  10683.478808  7.823337e+12   
max     8110.000

d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Epoch 30/100 - train_acc: 1.0000, val_acc: 1.0000
Early stopping at epoch 36

Test Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        10

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20


Confusion Matrix:
[[10  0]
 [ 0 10]]

Stratified Cross-Validation Scores: [0.5  0.5  0.5  0.4  0.55]
Mean CV Accuracy: 0.4900 (+/- 0.0980)

>>> Kaggle-format dataset: Path A test acc = 1.0000 | Path B mean CV = 0.4900

Ethan's cleaned dataset


d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.wa

Loaded dataset with shape: (810909, 6)
Columns: ['height', 'timestamp', 'size', 'tx_count', 'difficulty', 'output_amount']
Features used: ['blocks_mined', 'avg_transactions', 'avg_volume', 'difficulty', 'avg_fee', 'fee_volatility', 'avg_block_size', 'age', 'profitability']
Feature statistics:
       blocks_mined  avg_transactions    avg_volume    difficulty  \
count    100.000000        100.000000    100.000000  1.000000e+02   
mean    8109.090000       1114.228251  10404.839399  7.821774e+12   
std        0.287623          6.449839    354.259323  1.976015e+09   
min     8109.000000       1099.984708   9702.485099  7.817118e+12   
25%     8109.000000       1110.411148  10135.084353  7.820612e+12   
50%     8109.000000       1114.179060  10369.657438  7.821835e+12   
75%     8109.000000       1118.075256  10683.478808  7.823337e+12   
max     8110.000000       1129.204094  11403.203040  7.824585e+12   

            avg_fee  fee_volatility  avg_block_size         age  profitability  
cou

d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.91      1.00      0.95        10
           1       1.00      0.90      0.95        10

    accuracy                           0.95        20
   macro avg       0.95      0.95      0.95        20
weighted avg       0.95      0.95      0.95        20


Confusion Matrix:
[[10  0]
 [ 1  9]]

Stratified Cross-Validation Scores: [0.5  0.5  0.5  0.4  0.55]
Mean CV Accuracy: 0.4900 (+/- 0.0980)

>>> Ethan's cleaned dataset: Path A test acc = 0.9500 | Path B mean CV = 0.4900

BigQuery dataset


d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.wa

Loaded dataset with shape: (810909, 11)
Columns: ['number', 'timestamp', 'size', 'tx_count', 'bits', 'height', 'total_output_satoshis', 'output_amount', 'total_fee_satoshis', 'tx_count_check', 'difficulty']
Features used: ['blocks_mined', 'avg_transactions', 'avg_volume', 'difficulty', 'avg_fee', 'fee_volatility', 'avg_block_size', 'age', 'profitability']
Feature statistics:
       blocks_mined  avg_transactions    avg_volume    difficulty  \
count    100.000000        100.000000    100.000000  1.000000e+02   
mean    8109.090000       1114.224929  10404.815739  7.821704e+12   
std        0.287623          6.451771    354.243116  2.016031e+09   
min     8109.000000       1099.984708   9702.485099  7.817118e+12   
25%     8109.000000       1110.411148  10135.084353  7.820612e+12   
50%     8109.000000       1114.179060  10369.657438  7.821744e+12   
75%     8109.000000       1118.075256  10683.478808  7.823309e+12   
max     8110.000000       1129.204094  11403.203040  7.824585e+12   



d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.wa

In [5]:
print(f"\n{'='*60}\nSUMMARY\n{'='*60}")
for label, (test_acc, cv_mean) in results.items():
    print(f"{label:20s}  Path A: {test_acc:.4f}   Path B: {cv_mean:.4f}   Gap: {test_acc - cv_mean:+.4f}")


SUMMARY
Kaggle                Path A: 1.0000   Path B: 0.4900   Gap: +0.5100
Ethan cleaned         Path A: 0.9500   Path B: 0.4900   Gap: +0.4600
BigQuery              Path A: 1.0000   Path B: 0.4800   Gap: +0.5200


In [6]:
# ============================================================
# Fix the cross_val_score cloning bug (§78) and re-measure CV
#
# Bug: DNNModel constructs MLPClassifier with max_iter=1, warm_start=True.
# Real training happens via partial_fit() in his custom train() loop, which
# ignores max_iter. But cross_val_score() CLONES the estimator per fold, and
# a clone carries only constructor hyperparameters -- so each fold gets a
# fresh network trained by one ordinary .fit() call at max_iter=1.
# ============================================================
import warnings
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neural_network import MLPClassifier

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def find_mlp(obj):
    """Locate the MLPClassifier inside the DNNModel wrapper by introspection,
    so this cell doesn't silently break if the attribute name differs."""
    for name, value in vars(obj).items():
        if isinstance(value, MLPClassifier):
            return name, value
    raise AttributeError(
        f"No MLPClassifier attribute on {type(obj).__name__}. "
        f"Attributes present: {list(vars(obj))}"
    )


def cv_mean(estimator, X, y):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        return cross_val_score(estimator, X, y, cv=CV).mean()


def run_fixed_cv(csv_path, label):
    dp = DataProcessor(csv_path)
    df = dp.load_data()
    miners_df = dp.create_miner_simulation(df)
    X_train, X_test, y_train, y_test, features = dp.prepare_training_data(miners_df)

    X_full = np.vstack([X_train, X_test])
    y_full = np.concatenate([y_train, y_test])

    attr, base = find_mlp(DNNModel(input_shape=X_train.shape[1]))
    print(f"\n[{label}] MLPClassifier at DNNModel.{attr} -- "
          f"as constructed: max_iter={base.max_iter}, warm_start={base.warm_start}, "
          f"batch_size={base.batch_size}")

    variants = {
        # Reproduce the bug, so the comparison is like-for-like in this cell
        "broken (max_iter=1)": clone(base),

        # Fix A: match his actual training budget -- 100 epochs, batch 16.
        # This is the minimal honest fix: nothing changed but the epoch budget.
        "fixed A (100 ep, batch 16)": clone(base).set_params(
            max_iter=100, warm_start=False, batch_size=16),

        # Fix B: Fix A plus his early stopping (sklearn's accuracy-based
        # approximation of his loss-based version -- the §13 deviation).
        "fixed B (A + early stop)": clone(base).set_params(
            max_iter=100, warm_start=False, batch_size=16,
            early_stopping=True, validation_fraction=0.1, n_iter_no_change=20),

        # Fix C: generous budget, default batching. Just a check that the
        # result isn't limited by how long we let it train.
        # Results: same as the 100 max_iter, no effect
        "fixed C (max_iter=1000)": clone(base).set_params(
            max_iter=1000, warm_start=False),
    }

    scores = {name: cv_mean(est, X_full, y_full) for name, est in variants.items()}
    for name, score in scores.items():
        print(f"    {name:<28s} CV mean = {score:.4f}")
    return scores


fixed = {}
fixed["Kaggle"] = run_fixed_cv(KAGGLE_PATH, "Kaggle-format dataset")
fixed["Ethan cleaned"] = run_fixed_cv(ETHAN_CLEAN_PATH, "Ethan's cleaned dataset")
fixed["BigQuery"] = run_fixed_cv("bigquery_renamed_temp.csv", "BigQuery dataset")

print(f"\n{'='*88}\nCV BEFORE AND AFTER THE FIX (test accuracy from Path A, unchanged)\n{'='*88}")
print(f"{'dataset':<16s}{'test':>8s}{'broken':>10s}{'fix A':>10s}{'fix B':>10s}{'fix C':>10s}")
for lab, s in fixed.items():
    test_acc = results[lab][0]
    print(f"{lab:<16s}{test_acc:>8.4f}"
          + "".join(f"{v:>10.4f}" for v in s.values()))

Loaded dataset with shape: (810909, 13)
Columns: ['height', 'timestamp', 'size', 'tx_count', 'difficulty', 'median_fee_rate', 'avg_fee_rate', 'total_fees', 'fee_range_min', 'fee_range_max', 'input_count', 'output_count', 'output_amount']
Features used: ['blocks_mined', 'avg_transactions', 'avg_volume', 'difficulty', 'avg_fee', 'fee_volatility', 'avg_block_size', 'age', 'profitability']
Feature statistics:
       blocks_mined  avg_transactions    avg_volume    difficulty  \
count    100.000000        100.000000    100.000000  1.000000e+02   
mean    8109.090000       1114.228251  10404.839399  7.821774e+12   
std        0.287623          6.449839    354.259323  1.976015e+09   
min     8109.000000       1099.984708   9702.485099  7.817118e+12   
25%     8109.000000       1110.411148  10135.084353  7.820612e+12   
50%     8109.000000       1114.179060  10369.657438  7.821835e+12   
75%     8109.000000       1118.075256  10683.478808  7.823337e+12   
max     8110.000000       1129.204094  

In [7]:
# Test discrepancy between Ethan's cleaned dataset and kaggle dataset

# Is his training path deterministic on fixed inputs?
dp = DataProcessor(ETHAN_CLEAN_PATH)
X_train, X_test, y_train, y_test, _ = dp.prepare_training_data(
    dp.create_miner_simulation(dp.load_data()))

for i in range(5):
    m = DNNModel(input_shape=X_train.shape[1])
    m.train(X_train, y_train, epochs=100, batch_size=16)
    print(f"run {i+1}: test acc = {m.evaluate(X_test, y_test):.4f}")

Loaded dataset with shape: (810909, 6)
Columns: ['height', 'timestamp', 'size', 'tx_count', 'difficulty', 'output_amount']
Features used: ['blocks_mined', 'avg_transactions', 'avg_volume', 'difficulty', 'avg_fee', 'fee_volatility', 'avg_block_size', 'age', 'profitability']
Feature statistics:
       blocks_mined  avg_transactions    avg_volume    difficulty  \
count    100.000000        100.000000    100.000000  1.000000e+02   
mean    8109.090000       1114.228251  10404.839399  7.821774e+12   
std        0.287623          6.449839    354.259323  1.976015e+09   
min     8109.000000       1099.984708   9702.485099  7.817118e+12   
25%     8109.000000       1110.411148  10135.084353  7.820612e+12   
50%     8109.000000       1114.179060  10369.657438  7.821835e+12   
75%     8109.000000       1118.075256  10683.478808  7.823337e+12   
max     8110.000000       1129.204094  11403.203040  7.824585e+12   

            avg_fee  fee_volatility  avg_block_size         age  profitability  
cou

d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Epoch 30/100 - train_acc: 0.9844, val_acc: 0.7500
Early stopping at epoch 38

Test Accuracy: 0.9500

Classification Report:
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        10
           1       1.00      0.90      0.95        10

    accuracy                           0.95        20
   macro avg       0.95      0.95      0.95        20
weighted avg       0.95      0.95      0.95        20


Confusion Matrix:
[[10  0]
 [ 1  9]]
run 1: test acc = 0.9500
Training model with 80 samples...
Epoch 0/100 - train_acc: 0.5781, val_acc: 0.5000
Epoch 10/100 - train_acc: 0.8594, val_acc: 0.7500
Epoch 20/100 - train_acc: 0.9531, val_acc: 0.7500


d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Early stopping at epoch 25

Test Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        10

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20


Confusion Matrix:
[[10  0]
 [ 0 10]]
run 2: test acc = 1.0000
Training model with 80 samples...
Epoch 0/100 - train_acc: 0.5469, val_acc: 0.5625
Epoch 10/100 - train_acc: 0.8594, val_acc: 0.7500
Epoch 20/100 - train_acc: 0.9219, val_acc: 0.8750
Early stopping at epoch 22

Test Accuracy: 0.9500

Classification Report:
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        10
           1       1.00      0.90      0.95        10

    accuracy                           0.95        20
   macro avg       0.95      0.95      0.95        20
weighte

d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Epoch 10/100 - train_acc: 0.8438, val_acc: 0.8750
Epoch 20/100 - train_acc: 0.9219, val_acc: 0.8750
Epoch 30/100 - train_acc: 0.9844, val_acc: 0.8750
Early stopping at epoch 34

Test Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        10

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20


Confusion Matrix:
[[10  0]
 [ 0 10]]
run 4: test acc = 1.0000
Training model with 80 samples...


d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Epoch 0/100 - train_acc: 0.5781, val_acc: 0.5625
Epoch 10/100 - train_acc: 0.8438, val_acc: 0.8125
Epoch 20/100 - train_acc: 0.9062, val_acc: 0.9375
Epoch 30/100 - train_acc: 1.0000, val_acc: 0.9375
Early stopping at epoch 39

Test Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        10

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20


Confusion Matrix:
[[10  0]
 [ 0 10]]
run 5: test acc = 1.0000


In [8]:
# ============================================================
# Reduced feature set on Ethan's own datasets, through his own code.
#
# Design: 2x2 -- {full 9, reduced 7} x {broken CV, fix-A CV} -- plus
# Path A as a distribution rather than a single draw (§89).
#
# Why drop AFTER scaling: StandardScaler standardises each column
# independently, so scale-then-drop == drop-then-scale. His
# DataProcessor needs no modification.
#
# Why fix A is the informative column: broken CV trains one epoch from
# scratch per fold, so it sits at chance regardless of which features
# it gets. Bug 1 has to be held FIXED to see bug 2 at all.
# ============================================================
import io, contextlib

DROP = ["avg_transactions", "avg_volume"]
N_RUNS = 20

PATHS = {
    "Kaggle":        KAGGLE_PATH,
    "Ethan cleaned": ETHAN_CLEAN_PATH,
    "BigQuery":      "bigquery_renamed_temp.csv",
}


def silent(fn, *a, **kw):
    """train()/evaluate()/load_data() print per-epoch logs, describe() tables
    and a full classification report on every call. ~120 calls would bury
    the result."""
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **kw)


def build(csv_path):
    dp = DataProcessor(csv_path)
    df = silent(dp.load_data)
    miners_df = silent(dp.create_miner_simulation, df)
    X_train, X_test, y_train, y_test, features = silent(
        dp.prepare_training_data, miners_df)
    X_full = np.vstack([X_train, X_test])
    y_full = np.concatenate([y_train, y_test])
    return dict(X_full=X_full, y_full=y_full, X_train=X_train, y_train=y_train,
                X_test=X_test, y_test=y_test, features=list(features))


datasets = {name: build(p) for name, p in PATHS.items()}

# --- verify column order from the pipeline itself, not from memory ---
for name, d in datasets.items():
    assert len(d["features"]) == d["X_full"].shape[1] == 9, \
        f"{name}: {len(d['features'])} names vs {d['X_full'].shape[1]} columns"
    missing = [c for c in DROP if c not in d["features"]]
    assert not missing, f"{name}: {missing} not in {d['features']}"
print("feature order:", datasets["Kaggle"]["features"])
print("dropping:", DROP, "->",
      [datasets["Kaggle"]["features"].index(c) for c in DROP])


def path_a_dist(X_tr, y_tr, X_te, y_te, n_runs=N_RUNS):
    """Path A is nondeterministic (§89): two unseeded np.random calls inside
    train() that random_state=42 never reaches. Report the spread."""
    accs = []
    for _ in range(n_runs):
        m = DNNModel(input_shape=X_tr.shape[1])
        silent(m.train, X_tr, y_tr, epochs=100, batch_size=16)
        accs.append(float(silent(m.evaluate, X_te, y_te)))
    return np.array(accs)


rows = []
for name, d in datasets.items():
    keep = [i for i, f in enumerate(d["features"]) if f not in DROP]
    _, base = find_mlp(DNNModel(input_shape=len(keep)))   # width is irrelevant to CV clones

    for label, cols in [("full (9)", list(range(9))), ("reduced (7)", keep)]:
        Xf, Xtr, Xte = (d["X_full"][:, cols], d["X_train"][:, cols],
                        d["X_test"][:, cols])
        a = path_a_dist(Xtr, d["y_train"], Xte, d["y_test"])
        rows.append({
            "dataset": name, "features": label,
            "pathA_mean": a.mean(), "pathA_min": a.min(), "pathA_max": a.max(),
            "cv_broken": cv_mean(clone(base), Xf, d["y_full"]),
            "cv_fixA":   cv_mean(clone(base).set_params(
                max_iter=100, warm_start=False, batch_size=16), Xf, d["y_full"]),
        })

reduced_results = pd.DataFrame(rows)
print()
print(reduced_results.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

feature order: ['blocks_mined', 'avg_transactions', 'avg_volume', 'difficulty', 'avg_fee', 'fee_volatility', 'avg_block_size', 'age', 'profitability']
dropping: ['avg_transactions', 'avg_volume'] -> [1, 2]


d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.wa


      dataset    features  pathA_mean  pathA_min  pathA_max  cv_broken  cv_fixA
       Kaggle    full (9)      0.9850     0.9500     1.0000     0.4900   0.9000
       Kaggle reduced (7)      0.8850     0.8000     0.9500     0.5000   0.7500
Ethan cleaned    full (9)      0.9825     0.9000     1.0000     0.4900   0.9000
Ethan cleaned reduced (7)      0.8875     0.8000     0.9500     0.5000   0.7500
     BigQuery    full (9)      0.9400     0.8000     1.0000     0.4800   0.8900
     BigQuery reduced (7)      0.7925     0.7500     0.9000     0.5000   0.8000


In [9]:
seed_rows = []
for name, d in datasets.items():
    keep = [i for i, f in enumerate(d["features"]) if f not in DROP]
    _, base = find_mlp(DNNModel(input_shape=len(keep)))
    for label, cols in [("full (9)", list(range(9))), ("reduced (7)", keep)]:
        Xf = d["X_full"][:, cols]
        scores = []
        for s in [1, 3, 15, 17, 25, 29, 30, 36, 42, 50, 51, 67, 100]:   # §48's seeds
            est = clone(base).set_params(max_iter=100, warm_start=False,
                                         batch_size=16, random_state=s)
            folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=s)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConvergenceWarning)
                scores.append(cross_val_score(est, Xf, d["y_full"], cv=folds).mean())
        scores = np.array(scores)
        seed_rows.append({"dataset": name, "features": label,
                          "mean": scores.mean(), "std": scores.std(),
                          "min": scores.min(), "max": scores.max()})

print(pd.DataFrame(seed_rows).to_string(index=False,
                                        float_format=lambda v: f"{v:.4f}"))

      dataset    features   mean    std    min    max
       Kaggle    full (9) 0.9038 0.0182 0.8700 0.9400
       Kaggle reduced (7) 0.7485 0.0218 0.7100 0.7800
Ethan cleaned    full (9) 0.9038 0.0182 0.8700 0.9400
Ethan cleaned reduced (7) 0.7485 0.0218 0.7100 0.7800
     BigQuery    full (9) 0.9108 0.0287 0.8600 0.9600
     BigQuery reduced (7) 0.7523 0.0264 0.7100 0.8000


In [10]:
for name, p in PATHS.items():
    df = pd.read_csv(p, usecols=lambda c: c in ("height", "block_id", "number"))
    col = df.columns[0]
    s = df[col]
    print(f"{name:15s} col={col:9s} pre-sorted={s.is_monotonic_increasing} "
          f"index==position after sort: "
          f"{(df.sort_values(col).index == np.arange(len(df))).all()}")

Kaggle          col=height    pre-sorted=True index==position after sort: True
Ethan cleaned   col=height    pre-sorted=True index==position after sort: True
BigQuery        col=number    pre-sorted=True index==position after sort: True
